# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The dataset describes three female patients with non-classical 21-hydroxylase deficiency-associated infertility, including clinical features, hormone levels, imaging, and genetic variant tabulations.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Title:", metadata['name'])
print("Description:", metadata['description'])
print("Published Date:", metadata['datePublished'])
print("Authors (@id):", [a['@id'] for a in metadata['author']])
print("RecordSet (@id):", metadata.get('recordSet', []))

## 2. Data Overview
Review available record sets and their associated fields. All entities are referenced by their `@id`. In the dataset description, three tables are noted:
- Table 1: Clinical features
- Table 2: Hormone and imaging
- Table 3: Genetic variants

Let's enumerate the record sets and their fields (if available) by `@id`.

In [ ]:
# Fetch all RecordSet IDs from metadata
record_sets_metadata = metadata.get('recordSet', [])
print("Available RecordSets by @id:")
for rs in record_sets_metadata:
    print("-", rs)

# If RecordSets are empty, enumerate from dataset API
record_sets = dataset.record_sets
for rs in record_sets:
    print("RecordSet @id:", rs['@id'])
    print("  Name:", rs.get('name', '(unnamed)'))
    print("  Fields/Columns (@id):")
    for f in rs.get('field', []):
        print("    -", f['@id'], f.get('name', ''))
    for c in rs.get('column', []):
        print("    -", c['@id'], c.get('name', ''))
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Reference each record set by its `@id`.

Here we enumerate the RecordSet IDs programmatically and extract their contents. All columns within each RecordSet are also referenced by `@id`.

In [ ]:
# Collect all RecordSet @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("RecordSet IDs:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Here, we apply common processing steps on one sample record set.
- Filter by a numeric field (e.g., age, hormone level)
- Normalize numeric columns
- Group by a categorical variable (e.g., phenotype, clinical feature)

All references use `@id` for fields and columns. If column names are ambiguous, consult the previous overview cells.

In [ ]:
# Example: Choose RecordSet (first one) and columns
if record_set_ids:
    rs_id = record_set_ids[0]  # Use first RecordSet
    df = dataframes[rs_id]
    print(f"Analyzing RecordSet: {rs_id}")
    numeric_columns = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    print("Numeric columns:", numeric_columns)
    
    # If 'age at diagnosis', 'hormone level', or other is present, use it; otherwise pick any numeric
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by a categorical field if available
        group_columns = [col for col in df.columns if df[col].dtype == 'object']
        if group_columns:
            group_field_id = group_columns[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Visualize distributions of numeric fields or relationships between variables using matplotlib or seaborn. Use the RecordSet and field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot a histogram and boxplot for the numeric attribute in the main record set
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    numeric_columns = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        sns.histplot(df[numeric_field_id], kde=True)
        plt.title(f"Distribution of {numeric_field_id}")

        plt.subplot(1, 2, 2)
        sns.boxplot(x=df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id}")
        plt.tight_layout()
        plt.show()
        
        # Scatterplot for two numeric fields if present
        if len(numeric_columns) >= 2:
            plt.figure(figsize=(6,4))
            sns.scatterplot(x=df[numeric_columns[0]], y=df[numeric_columns[1]])
            plt.xlabel(numeric_columns[0])
            plt.ylabel(numeric_columns[1])
            plt.title(f"Scatterplot of {numeric_columns[0]} vs {numeric_columns[1]}")
            plt.show()

## 6. Conclusion
This notebook demonstrated loading and processing the FAIR^2 dataset using `mlcroissant` referencing the schema and all entities by `@id`. Key steps included:
- Loading via Croissant schema URL
- Enumerating RecordSets and fields (`@id`)
- Extracting and processing record set data
- Performing basic EDA and visualization

Further analysis can be performed on hormone levels, phenotypes, and genetic variants, referencing their column `@id`s for robust and reproducible workflows. See dataset documentation for context on clinical and genetic heterogeneity.